# importing libs and pkgs

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms.v2 as v2

from torch.utils.data import Dataset , DataLoader
from torch.random import Generator
device = "cuda" if torch.cuda.is_available() else "cpu"

import numpy as np
np.random.seed(58)
import pandas as pd
import matplotlib.pyplot as plt

import kagglehub
#kagglehub.login()
from tqdm import tqdm

import os
from PIL import Image
import pickle

In [ ]:
!pip install segmentation-models-pytorch

In [ ]:
import segmentation_models_pytorch as smp

# importing and creating the dataset

In [ ]:
# blood dataset
path = kagglehub.dataset_download("jeetblahiri/bccd-dataset-with-mask")

print("Path to dataset files:", path)


Path to dataset files: /kaggle/input/bccd-dataset-with-mask


In [ ]:
path = kagglehub.dataset_download("rajkumarl/people-clothing-segmentation")
print(path)

/kaggle/input/people-clothing-segmentation


In [ ]:
class Bccd_Dataset(Dataset):
  def __init__(self , transform , target_transform ):
    pass

In [ ]:
class people_clothing_segmentation(Dataset):
  def __init__(self , transform , target_transform ):
    #jpeg_img_path = "/kaggle/input/people-clothing-segmentation/jpeg_images/IMAGES/"
    #jpeg_masks_path = "/kaggle/input/people-clothing-segmentation/jpeg_masks/MASKS/"

    png_img_path = "/kaggle/input/people-clothing-segmentation/png_images/IMAGES/"
    png_masks_path = "/kaggle/input/people-clothing-segmentation/png_masks/MASKS/"

    #jpeg_images = os.listdir(jpeg_img_path)
    #jpeg_masks = os.listdir(jpeg_masks_path)

    png_images = os.listdir(png_img_path)
    png_masks = os.listdir(png_masks_path)

    #self.images = [jpeg_img_path + i for i in jpeg_images]
    self.images = [png_img_path + i for i in png_images]
    #self.masks = [jpeg_masks_path + i for i in jpeg_masks]
    self.masks = [png_masks_path + i for i in png_masks]

    self.images.sort()
    self.masks.sort()

    self.data = list(zip(self.images , self.masks))

    self.transform = transform
    self.target_transform = target_transform

  def __len__(self):
    return len(self.data)

  def __getitem__(self,idx):

    img_path , mask_path = self.data[idx]
    img = Image.open(img_path)
    mask = Image.open(mask_path)

    img = self.transform(img)
    img = img


    mask = self.target_transform(mask)




    return img , mask


In [ ]:
labels = pd.read_csv('/kaggle/input/people-clothing-segmentation/labels.csv')
labels1 = pd.read_csv('/kaggle/input/people-clothing-segmentation/labels (1).csv')

In [ ]:
pd.concat([labels,labels1] ,axis=1)

,Unnamed: 0,label_list,Unnamed: 0,label_list
0,0,NaN,0,NaN
1,1,accessories,1,accessories
2,2,bag,2,bag
3,3,belt,3,belt
4,4,blazer,4,blazer
5,5,blouse,5,blouse
6,6,bodysuit,6,bodysuit
7,7,boots,7,boots
8,8,bra,8,bra
9,9,bracelet,9,bracelet


# hyperparams


In [ ]:
batch_size = 16
img_size = (640,360)
target_img_size = (640,360)
n_cls = 59 # number of classes in the dataset

residual = False


transform = v2.Compose([
    v2.Resize(img_size),
    #v2.RandomCrop((640,360)),
    v2.ToImage(),
    v2.ToDtype(torch.float32,scale=True),



])

target_transform = v2.Compose([
    v2.Resize(target_img_size),
    v2.ToImage(),
    v2.ToDtype(dtype=torch.long),
])


# dataset and dataloaders

In [ ]:
dataset = people_clothing_segmentation(transform=transform,target_transform=target_transform)

train_set , val_set  , test_set = torch.utils.data.random_split(dataset ,
                                                                [0.75,0.15,0.1] ,
                                                                torch.Generator().manual_seed(58) )


train_loader = DataLoader(train_set , batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set , batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_set , batch_size=batch_size, shuffle=True)


# creating the models

1. U-net
2. segnet
3. pspnet
4. custom arch if there is a time (it's idea I have)



### U-net

##### defining multiple model blocks

In [ ]:
class ClassicBlock(nn.Module):
  def __init__(self,in_channels , out_channels , kernel_size=3 , padding = 1):
    super(ClassicBlock, self).__init__()
    self.conv1 = nn.Conv2d(in_channels , out_channels , kernel_size=kernel_size , padding =padding)
    self.conv2 = nn.Conv2d(out_channels , out_channels , kernel_size=kernel_size , padding = padding)
    self.batchnorm = nn.BatchNorm2d(out_channels)

  def forward(self,x):
    x = self.conv1(x)
    x = F.relu(x)
    x = self.conv2(x)
    x = self.batchnorm(x)
    x = F.relu(x)

    return x

class ResidualBlock(nn.Module):
  def __init__(self,in_channels , out_channels , kernel_size=3 , padding =1):
    super(ResidualBlock, self).__init__()
    self.conv1 = nn.Conv2d(in_channels , out_channels , kernel_size=kernel_size ,padding=padding)
    self.conv2 = nn.Conv2d(out_channels , out_channels , kernel_size=kernel_size , padding=padding)
    self.batchnorm = nn.BatchNorm2d(out_channels)

    self.downsample = nn.Conv2d(in_channels , out_channels , kernel_size=1,padding=0)

  def forward(self,x):

    residual = x
    x = self.conv1(x)
    x = F.relu(x)
    x = self.conv2(x)
    x = self.batchnorm(x)
    x = F.relu(x)

    if residual.shape != x.shape:
      residual = self.downsample(residual)

    x = x + residual
    x = F.relu(x)

    return x


class Channel_Attention(nn.Module):
    def __init__(self , in_channels,reduction_ratio):
        super(Channel_Attention , self).__init__()
        self.shared_mlp = nn.Sequential(
            nn.Linear(in_channels , in_channels // reduction_ratio),
            nn.ReLU(),
            nn.Linear(in_channels // reduction_ratio , in_channels)

        )



    def forward(self , x):
        feature_map = x
        b , c , h , w = x.shape

        maxpool = F.max_pool2d(x , kernel_size=(h,w))
        avgpool = F.avg_pool2d(x, kernel_size=(h,w))

        maxpool = self.shared_mlp(maxpool.view(b,-1))
        avgpool = self.shared_mlp(avgpool.view(b,-1))

        pooled_features = (maxpool + avgpool).unsqueeze(2)

        return F.sigmoid(pooled_features).unsqueeze(2) * feature_map




class Spatial_Attention(nn.Module):
    def __init__(self , kernel_size = 7 , padding = 3):
        super(Spatial_Attention , self).__init__()

        self.shared_conv = nn.Conv2d(2,1,kernel_size=(7,7) ,padding=3)


    def forward(self, x ):
        feature_map = x

        b,c,h,w = x.shape

        channel_maxpool = F.max_pool3d(x , (c,1,1))
        channel_avgpool = F.avg_pool3d(x , (c,1,1))

        spatial_descriptor = torch.cat([channel_avgpool,channel_maxpool] , dim = 1)

        spatial_descriptor = self.shared_conv(spatial_descriptor)

        spatial_descriptor = F.sigmoid(spatial_descriptor)

        return spatial_descriptor * feature_map





class CBAM(nn.Module):
    def __init__(self,in_channels , reduction_ratio = 2 , resnet = None):
        super(CBAM , self ).__init__()

        self.resnet = resnet

        self.channel_attention = Channel_Attention(in_channels , reduction_ratio)
        self.spatial_attention = Spatial_Attention()



    def forward(self , x):
        if self.resnet:
            F = x

            x = self.channel_attention(x)
            x = self.spatial_attention(x)

            return x + F
        else :

            x = self.channel_attention(x)
            x = self.spatial_attention(x)

            return (x)





class Encoder(nn.Module):
  def __init__(self,in_channels , out_channels , kernel_size=3 ,padding = 0 , model_block = None):
    super(Encoder, self).__init__()

    self.model_block = model_block

    if self.model_block == "residualBlock" :
      self.Down = ResidualBlock(in_channels , out_channels , kernel_size )

    elif self.model_block == "CBAM":
      self.cbam = CBAM(in_channels=out_channels , resnet=None)
      self.Down = ClassicBlock(in_channels , out_channels , kernel_size , padding=padding)

    elif self.model_block == "CBAM_resnet":
      self.cbam = CBAM(in_channels=out_channels , resnet=True)
      self.Down = ResidualBlock(in_channels , out_channels , kernel_size , padding=padding)

    else :
      self.Down = ClassicBlock(in_channels , out_channels , kernel_size , padding=padding)

    self.pool = nn.MaxPool2d(2,2)

  def forward(self,x):

    x = self.Down(x)

    if self.model_block == "CBAM" or self.model_block == "CBAM_resnet":
      feature_map = self.cbam(x)
    else :
      feature_map = x


    x = self.pool(x)

    return x , feature_map

class Decoder(nn.Module):
  def __init__(self,in_channels , out_channels , kernel_size=3 ,scale=2, padding = 0 , model_block = None):
    super(Decoder, self).__init__()
    self.up = nn.Upsample(scale_factor=scale , mode='bilinear' , align_corners=True)
    if model_block == "residualBlock" :
      self.upConv = ResidualBlock(in_channels + out_channels, out_channels , kernel_size )
    else :
      self.upConv = ClassicBlock(in_channels + out_channels, out_channels , kernel_size , padding=padding)


  def forward(self,x,feature_map):

    x = self.up(x)

    diffy = torch.tensor(feature_map.size()[2] - x.size()[2])
    diffx = torch.tensor(feature_map.size()[3] - x.size()[3])

    x = nn.functional.pad(x, (diffx//2, diffx-diffx//2,
                                    diffy//2, diffy-diffy//2))
    x = torch.cat([x, feature_map], dim=1)


    x = self.upConv(x)

    return x


  def _center_crop(self , tensor, target_tensor):
    _, _, h, w = target_tensor.shape
    tensor = v2.CenterCrop([h, w])(tensor)
    return tensor



In [ ]:
class UNet(torch.nn.Module):
    def __init__(self, model_block = None , padding=1 , classes=n_cls):
        super(UNet, self).__init__()
        self.model_block = model_block
        self.encoder1 = Encoder(3,64 , model_block=model_block ,padding=padding)
        self.encoder2 = Encoder(64,128 , model_block=model_block ,padding=padding)
        self.encoder3 = Encoder(128,256, model_block=model_block ,padding=padding)
        self.encoder4 = Encoder(256,512 , model_block=model_block, padding=padding)
        self.center = ClassicBlock(512,1024 )
        self.decoder4 = Decoder(1024,512 , model_block=model_block, padding=padding)
        self.decoder3 = Decoder(512,256 , model_block=model_block , padding=padding)
        self.decoder2 = Decoder(256,128, model_block=model_block,padding=padding)
        self.decoder1 = Decoder(128,64 ,model_block=model_block,padding=padding)
        self.final = nn.Conv2d(64,n_cls,kernel_size=1)
        self.residual_center = nn.Sequential(
            nn.Conv2d(512,1024,kernel_size=1),
            nn.BatchNorm2d(1024),
            nn.ReLU()
        )
        self.drop = nn.Dropout(p=0.2,inplace=True)

    def forward(self,x):

        x, feature_map1 = self.encoder1(x)
        x, feature_map2 = self.encoder2(x)
        x, feature_map3 = self.encoder3(x)
        x, feature_map4 = self.encoder4(x)

        if self.model_block == "residualBlock":
          residual = x
          x = self.center(x)
          residual = self.residual_center(residual)
          x = residual + x
        else :
          x = self.center(x)

        #feature_map4 = self.drop(feature_map4)
        #feature_map3 = self.drop(feature_map3)
        #feature_map2 = self.drop(feature_map2)
        #feature_map1 = self.drop(feature_map1)

        x = self.decoder4(x,feature_map4)
        x = self.decoder3(x,feature_map3)
        x = self.decoder2(x,feature_map2)
        x = self.decoder1(x,feature_map1)
        x = self.final(x)

        return x





In [ ]:
def compute_metrics(pred, target, num_classes):
    # Flatten tensors
    pred = pred.argmax(dim=1)
    target = target.squeeze(1)

    pa = (pred == target).sum().float() / target.numel()

    ious = []
    for cls in range(num_classes):
        pred_inds = pred == cls
        target_inds = target == cls

        intersection = (pred_inds & target_inds).sum().float()
        union = (pred_inds | target_inds).sum().float()

        if union == 0:
            ious.append(torch.tensor(float('nan')))  # or 0
        else:
            ious.append(intersection / union)

    mean_iou = torch.tensor(ious).nanmean()  # Ignore NaNs for empty classes
    return pa.item(), mean_iou.item(), ious


In [ ]:
def dice_loss(pred, target, smooth=1.0):
    # Apply sigmoid if not already
    pred = torch.softmax(pred , dim=1)

    # Flatten
    pred = pred.contiguous().view(-1)
    target = target.contiguous().view(-1)

    intersection = (pred * target).sum()
    dice = (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)

    return 1 - dice

def multiclass_dice_loss(pred, target, num_classes, smooth=1.0):
    pred = F.softmax(pred, dim=1)

    total_loss = 0
    for c in range(num_classes):
        pred_c = pred[:, c]
        target_c = (target == c).float()
        intersection = (pred_c * target_c).sum()
        dice = (2. * intersection + smooth) / (pred_c.sum() + target_c.sum() + smooth)
        total_loss += 1 - dice

    return total_loss / num_classes



In [ ]:
cbam_unet = UNet(model_block = 'CBAM_resnet')

In [ ]:
cbam_unet.to(device)

UNet(
  (encoder1): Encoder(
    (cbam): CBAM(
      (channel_attention): Channel_Attention(
        (shared_mlp): Sequential(
          (0): Linear(in_features=64, out_features=32, bias=True)
          (1): ReLU()
          (2): Linear(in_features=32, out_features=64, bias=True)
        )
      )
      (spatial_attention): Spatial_Attention(
        (shared_conv): Conv2d(2, 1, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3))
      )
    )
    (Down): ResidualBlock(
      (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (batchnorm): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (downsample): Conv2d(3, 64, kernel_size=(1, 1), stride=(1, 1))
    )
    (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (encoder2): Encoder(
    (cbam): CBAM(
      (channel_attention): Channel_Attention(
        (share

In [ ]:
model = smp.Unet(classes=n_cls,encoder_weights='imagenet')
model.to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Unet(
  (encoder): ResNetEncoder(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track

In [ ]:
cbam_unet(torch.rand((16,3,640,360)).to(device)).shape

torch.Size([16, 59, 640, 360])

In [ ]:
params = model.parameters()
optimizer = torch.optim.Adam(params ,lr=9e-4)
loss_fn = smp.losses.DiceLoss('multiclass')

In [ ]:
def train(model , optimizer ,  train_loader , val_loader , loss_fn=loss_fn, num_epochs=10 , early_stopping=5):


  train_losses , pixel_acc_tr , iou_tr = [] , [] , []
  val_losses , pixel_acc_val , iou_val = [] , [] , []

  met_tr = []
  met_val = []

  patince_count = 0


  best_val_loss = float('inf')

  for i in range(num_epochs):
    print()
    print(f'epochs No:{i+1}/{num_epochs}')
    print("---------------")
    train_loss , tr_pa , tr_iou = 0 , 0 , 0
    val_loss , val_pa , val_iou = 0 , 0 , 0

    model.train()
    for batch in train_loader:
      inputs , masks = batch


      inputs = inputs.to(device)
      masks = masks.to(device)

      optimizer.zero_grad()
      outputs = model(inputs)



      #met_res = compute_metrics(outputs, masks, n_cls)


      loss = loss_fn(outputs,masks) #multiclass_dice_loss(outputs, masks.squeeze(1).long() ,n_cls)
      loss.backward()
      optimizer.step()

      train_loss += loss.item()

      outputs = outputs.argmax(dim=1)
      #masks = masks.round().long()


      tp,fp,fn,tn = smp.metrics.get_stats(outputs, masks.squeeze(1), mode='multiclass', num_classes=n_cls)

      tr_pa += smp.metrics.accuracy(tp,fp,fn,tn,reduction='micro').item()

      tr_iou += smp.metrics.iou_score(tp, fp, fn, tn, reduction="micro").item()


    print(f'train loss : {train_loss / len(train_loader)}')
    print(f'train pa : {tr_pa / len(train_loader)}')
    print(f'train iou : {tr_iou / len(train_loader)}')
    pixel_acc_tr.append(tr_pa / len(train_loader))
    iou_tr.append(tr_iou / len(train_loader))
    train_losses.append(train_loss / len(train_loader))
    #met_tr.append(met_res)

    model.eval()
    with torch.no_grad():
      for batch in val_loader:
        inputs , masks = batch

        inputs = inputs.to(device)

        masks = masks.to(device)

        outputs = model(inputs)

        #met_res = compute_metrics(outputs, masks, n_cls)

        loss = loss_fn(outputs,masks)  #multiclass_dice_loss(outputs, masks.squeeze(1).long() ,n_cls)




        val_loss += loss.item()


        outputs = outputs.argmax(dim=1)
        #masks = masks.round().long()

        tp,fp,fn,tn = smp.metrics.get_stats(outputs, masks.squeeze(1), mode='multiclass', num_classes=n_cls)



        val_iou += smp.metrics.iou_score(tp, fp, fn, tn, reduction="micro").item()
        val_pa += smp.metrics.accuracy(tp ,fp,fn,tn , reduction='micro').item()



      if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict() , 'best_model.pt')
        patince_count = 0
      else :
        if patince_count > early_stopping:
          break
        patince_count += 1


      print()
      print(f'val loss : {val_loss/len(val_loader)}')
      print(f'val pa : {val_pa / len(val_loader)}')
      print(f'val iou : {val_iou / len(val_loader)}')

      pixel_acc_val.append(val_pa / len(val_loader))
      iou_val.append(val_iou / len(val_loader))
      val_losses.append(val_loss / len(val_loader))
      #met_val.append(met_res)

    print("---------------")




  return (train_losses,pixel_acc_tr,iou_tr) , (val_losses , pixel_acc_val , iou_val)



In [ ]:
train_metrics , val_metrics = train(model , optimizer  , train_loader, val_loader ,num_epochs=50)


epochs No:1/50
---------------
train loss : 0.9360134043592088
train pa : 0.9685136663152817
train iou : 0.03710223926587942

val loss : 0.9187704443931579
val pa : 0.9694676995277405
val iou : 0.052262387424707415
---------------

epochs No:2/50
---------------
train loss : 0.8964354991912842
train pa : 0.9738893876684472
train iou : 0.14375882001316292

val loss : 0.9099252223968506
val pa : 0.9826125621795654
val iou : 0.3220161646604538
---------------

epochs No:3/50
---------------
train loss : 0.8723502818574297
train pa : 0.9926748415257068
train iou : 0.6532369148223958

val loss : 0.8863669037818909
val pa : 0.9937860548496247
val iou : 0.6903096199035644
---------------

epochs No:4/50
---------------
train loss : 0.8589900224766833
train pa : 0.9943405377103928
train iou : 0.7140340132916227

val loss : 0.8755569398403168
val pa : 0.9935631394386292
val iou : 0.6809498906135559
---------------

epochs No:5/50
---------------
train loss : 0.839853805430392
train pa : 0.9944

KeyboardInterrupt: 

In [ ]:
def test(model , test_loader):
  model.eval()
  test_losses = np.array([])

  test_ious = np.array([])
  test_pas = np.array([])



  for i in range(5):
    test_loss = 0
    test_iou = 0
    test_pa = 0
    with torch.no_grad():
      for batch in test_loader:
        inputs , masks = batch

        inputs = inputs.to(device)

        masks = masks.to(device)

        outputs = model(inputs)

        loss = loss_fn(outputs,masks)



        outputs = outputs.argmax(dim=1)
        masks = masks.squeeze(1)


        test_loss += loss.item()

        tp,fp,fn,tn = smp.metrics.get_stats(outputs, masks.squeeze(1), mode='multiclass', num_classes=n_cls)



        test_iou += smp.metrics.iou_score(tp, fp, fn, tn, reduction="micro").item()
        test_pa += smp.metrics.accuracy(tp ,fp,fn,tn , reduction='micro').item()

    test_losses = np.append(test_losses , test_loss / len(test_loader))
    test_ious = np.append(test_ious , test_iou / len(test_loader))
    test_pas = np.append(test_pas , test_pa / len(test_loader))

  return test_losses.mean() , test_ious.mean() , test_pas.mean()

In [ ]:
test_loss , test_iou , test_pa = test(cbam_unet , test_loader)

In [ ]:
print(test_loss , test_iou , test_pa)

0.7956475104604449 0.7142139860561916 0.9943442566054207


In [ ]:
metrics = {
    "train_loss" : list(train_metrics[0]),
    "train_pa" : list(train_metrics[1]),
    "train_iou" : list(train_metrics[2]),
    "val_loss" : list(val_metrics[0]),
    "val_pa" : list(val_metrics[1]),
    "val_iou" : list(val_metrics[2]),
    "test_loss" : test_loss,
    "test_pa" : test_pa,
    "test_iou" : test_iou
}





In [ ]:
with open("cbam_resnet_unet_metrics_.pkl" , "wb") as pkl :
  pickle.dump(metrics , pkl)

In [ ]:
batch = next(iter(train_loader))

In [ ]:
len(val_loader) * val_loader.batch_size

100

In [ ]:
batch[0].shape

torch.Size([16, 3, 640, 360])

In [ ]:
out = model(batch[0].to(device))

In [ ]:
F.cross_entropy(out,batch[1].to(device).squeeze(1).long())

tensor(2.9104e-13, device='cuda:0', grad_fn=<NllLoss2DBackward0>)

In [ ]:
match = torch.eq(out.to('cpu') , batch[1])

In [ ]:
torch.argmax(batch[1] , dim = 1).shape

torch.Size([16, 640, 360])

In [ ]:
torch.numel(out.squeeze(1))

217497600

In [ ]:
torch.cuda.empty_cache()

In [ ]:
img = v2.ToPILImage()(batch[0][1])

In [ ]:
tar = v2.ToPILImage()(out[1].argmax(dim=0).int())

In [ ]:
img

NameError: name 'img' is not defined

In [ ]:
tar.type

NameError: name 'tar' is not defined

In [ ]:
target = torch.randint(0, 58, (16, 4, 4))

In [ ]:
target.shape

torch.Size([16, 4, 4])

In [ ]:
torch.argmax(out[0],dim=0).unique()

tensor([0], device='cuda:0')

In [ ]:
torch.numel(out.squeeze(1))

112614192

In [ ]:
out[12].argmax(dim=0).unique()

tensor([ 0,  1,  2,  3,  4,  5,  6,  8,  9, 10, 11, 13, 15, 16, 17, 18, 19, 20,
        21, 22, 23, 26, 28, 29, 30, 31, 32, 33, 34, 35, 38, 39, 41, 48, 49, 50,
        53, 54, 55, 56, 57], device='cuda:0')

In [ ]:
batch[1].squeeze(1).shape

torch.Size([18, 404, 267])

In [ ]:
with open('first30_eps_big_metrics.pkl' , 'wb') as f:
  pickle.dump((train_metrics , val_metrics ) , f)


In [ ]:
resblock = ResidualBlock(3,64,padding=1)

In [ ]:
resblock(torch.rand(3*816*544).view(1,3,816,544))

torch.Size([1, 64, 816, 544]) torch.Size([1, 64, 816, 544])


tensor([[[[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.2026],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0710, 1.5576],
          [0.0000, 0.0000, 0.7663,  ..., 0.0000, 0.2153, 1.3460],
          ...,
          [0.0000, 0.0000, 0.0000,  ..., 0.6899, 0.3646, 0.0181],
          [0.0000, 0.0000, 0.5030,  ..., 0.0000, 0.0000, 0.4004],
          [0.0000, 0.0000, 0.8083,  ..., 0.4510, 0.0273, 1.3079]],

         [[0.3343, 1.3094, 1.2863,  ..., 1.0498, 1.7026, 1.3807],
          [0.4243, 0.8606, 0.3974,  ..., 0.6883, 1.3168, 0.3489],
          [0.2526, 0.4810, 0.6520,  ..., 1.3870, 1.3335, 1.3324],
          ...,
          [0.1988, 1.8533, 1.5305,  ..., 0.1514, 0.4740, 1.1873],
          [0.5211, 1.3129, 1.1065,  ..., 0.3786, 1.5375, 1.0520],
          [0.5507, 0.4115, 0.2935,  ..., 0.4107, 0.4826, 0.6374]],

         [[0.6713, 0.6380, 0.7065,  ..., 0.8757, 0.8095, 0.4911],
          [0.9049, 0.7876, 1.8461,  ..., 1.9634, 0.5799, 2.2476],
          [0.7213, 1.0222, 0.8474,  ..., 1

In [ ]:
t1 = torch.rand(1,3,16,16)
t2 = torch.rand(1,64,16,16)
torch.sum(t1,t2 , dim =0)

TypeError: sum() received an invalid combination of arguments - got (Tensor, Tensor, dim=int), but expected one of:
 * (Tensor input, *, torch.dtype dtype = None)
      didn't match because some of the keywords were incorrect: dim
 * (Tensor input, tuple of ints dim, bool keepdim = False, *, torch.dtype dtype = None, Tensor out = None)
 * (Tensor input, tuple of names dim, bool keepdim = False, *, torch.dtype dtype = None, Tensor out = None)


In [ ]:
smp.metrics.accuracy()

TypeError: accuracy() missing 4 required positional arguments: 'tp', 'fp', 'fn', and 'tn'

In [ ]:
smp.metrics.get_stats()

TypeError: get_stats() missing 3 required positional arguments: 'output', 'target', and 'mode'

In [ ]:
type(train_metrics[0])

list

In [ ]:
metrics["test_iou"]

np.float64(0.7057273983955383)